# GridLeak AI — Model Eğitimi ve SMOTE Karşılaştırması

Mevcut master defterin hücreleri içerikleri ve kayıtlı çıktıları korunarak bu deftere taşınmıştır. Ortak hazırlık hücreleri bağımsız açılış için tekrar yer alır.

Bu düzenleme yeni eğitim veya değerlendirme çalıştırmamıştır. Hücreleri çalıştırmak ilgili hesaplamaları yeniden yürütür.

## Tekrar çalıştırılabilir seçim ve hiperparametre araması
Kod: `ai/training/research_pipeline.py`.
1. Ham tüketimden eğitim abonelerini ve 49 temel özelliği oluşturur.
2. Her CV katında bağlam/peer/SVD'yi yalnız öğrenme grubuna fit eder.
3. Doğrulama gruplarında TreeSHAP ve permutation AP hesaplayıp adayları sıralar.
4. LightGBM, XGBoost, CatBoost ve MLP için ayrı Optuna TPE aramaları yürütür.
5. Her denemeyi AP, F1, Precision, Recall, FP/FN ile kaydeder.

Bu eklenen arama, geçmişteki Optuna oturumunun kurtarılmış kaydı değildir.
Kilitli final v3 parametreleri `ai/training/v3_reference.json` içinde ayrıca korunur.
Yeni sonuçlar yeni deney kimliğindedir. Seçim ve ayarlamada tekrar kullanılan OOF,
geliştirme skorudur; yeni bağımsız dış doğrulama diye sunulmaz. F1 eşiği aynı
OOF üzerinde seçilir. Test dosyaları araştırma modülü tarafından kullanılmaz.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'ai/features/prepare_data.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import subprocess
RUN_RESEARCH = False
if RUN_RESEARCH:
    from datetime import datetime
    output = ROOT/'ai/artifacts'/('research_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    subprocess.run([sys.executable, str(ROOT/'ai/training/research_pipeline.py'),
                    '--output', str(output), '--mode', 'all', '--trials', '20'], check=True)
else:
    print('Seçim ve dört aile için Optuna araması hazır; pahalı deney otomatik başlatılmadı.')


In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
warnings.filterwarnings('ignore')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ai/features/v3_features.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Projedeki GERÇEK üretim modüllerini doğrudan içe aktarıyoruz (sıfır kod tekrarı!)
from ai.features.v3_features import load_raw_for_ids, extract_subscriber_features, FoldContextTransformer
from ai.training.v3_final_train_and_test import build_base_estimators, build_meta_estimator, ece_score

EXP = ROOT / 'ai/artifacts/iyilestirme_v3_20261123T171349Z'
V3 = ROOT / 'data/processed/v3'
lock = json.loads((EXP / 'locked_selection.json').read_text(encoding='utf-8'))
fcols = lock['features']
print('Proje Kökü:', ROOT)
print('Aktif Kilitli Model:', lock['model_name'])
print('Kilitli Özellik Sayısı:', len(fcols))

Proje Kökü: c:\Users\kadir\GridLeakAI
Aktif Kilitli Model: Calibrated Stacking v3 (LGBM + CatBoost + XGBoost + MLP)
Kilitli Özellik Sayısı: 160


## 1. Veri Seti Yapısı, %80/%20 Abone Bölünmesi ve Sınıf Dengesizliği
Aboneler kimlik (`CONS_NO`) bazında tabakalı bölünmüştür; eğitim ve test arasında **sıfır ortak abone** vardır.

In [ ]:
# Geliştirme aşamasında yalnız eğitim verisi yüklenir.
train_df = pd.read_csv(V3 / 'egitim_ozellikleri_2016_v1.csv', dtype={'CONS_NO': str})
y_tr = train_df['FLAG'].to_numpy(int)
print('Eğitim:', len(train_df), '| FLAG=1:', int(y_tr.sum()))


## SMOTE: bağlam dönüşümleri de kat içinde öğrenilir
Yeni karşılaştırma `research_pipeline.py` içindedir. Şebeke/peer/SVD yalnız
öğrenme abonelerine fit edilir. Eksik doldurma ve ölçekleme iki kolda aynıdır;
tek fark öğrenme katına eklenen SMOTE'dur. Doğrulama örnekleri çoğaltılmaz.

Önceki kaydedilmiş tablo, tüm eğitim verisinden hazırlanmış bağlam özelliklerini
kullandığından tarihsel keşif sonucudur; yeni kat-içi doğrulama sonucu değildir.
Yeni sonuçlar yalnız aşağıdaki akış gerçekten çalıştırıldığında oluşur.


In [ ]:
from ai.training.research_pipeline import prepare_folds, compare_smote
RUN_SMOTE = False
if RUN_SMOTE:
    from datetime import datetime
    output = ROOT/'ai/artifacts'/('smote_recheck_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    prepare_folds(ROOT/'data/raw/data.csv', ROOT/'data/splits/abone_bolunmesi.csv', output)
    display(compare_smote(output, fcols))
else:
    print('Kat-içi SMOTE hazır. Çalıştırılmadı; final model değiştirilmedi.')


## 4. 4 Aileli Kalibre Stacking Mimarisi ve Bağımsız 5-Fold OOF Sonuçları
Eğitilmiş final modeli (`ensemble_v3.joblib`) inceleyerek içindeki 4 farklı aileyi (**LightGBM, CatBoost, XGBoost, RankGauss MLP**) ve logit-uzayı Sigmoid/Platt meta-öğrenicisini doğruluyoruz.

In [5]:
model = joblib.load(EXP / 'ensemble_v3.joblib')
print('Temel Model Aileleri:', list(model.named_estimators_.keys()))
print('Meta-Öğrenici (Kalibrasyon):', model.final_estimator_)
print('decision_function var mı? (False olmalı):', hasattr(model, 'decision_function'))

cv_df = pd.read_csv(EXP / 'model_karsilastirma.csv')
display(cv_df)

Temel Model Aileleri: ['lgbm', 'catboost', 'xgboost', 'mlp']
Meta-Öğrenici (Kalibrasyon): Pipeline(steps=[('clip',
                 FunctionTransformer(func=<function clip at 0x0000020B4E60BDF0>,
                                     kw_args={'a_max': 0.999999,
                                              'a_min': 1e-06})),
                ('logit', FunctionTransformer(func=<ufunc 'logit'>)),
                ('cal',
                 CalibratedClassifierCV(cv=3,
                                        estimator=LogisticRegression(C=0.5,
                                                                     max_iter=1000)))])
decision_function var mı? (False olmalı): False


,Model,CV_Folds,OOF_AP,OOF_ROC_AUC,OOF_F1,OOF_Precision,OOF_Recall,OOF_FP,OOF_FN
0,lgbm,5,0.63275,0.90780,0.5801,0.6182,0.5465,966,1298
1,catboost,5,0.62082,0.90091,0.5738,0.6041,0.5465,1025,1298
2,xgboost,5,0.61942,0.90413,0.5632,0.5915,0.5374,1062,1324
3,mlp,5,0.50656,0.84909,0.4845,0.4803,0.4888,1514,1463
4,Calibrated Stacking v3 (LGBM + CatBoost + XGBo...,5,0.63565,0.90847,0.5775,0.6068,0.5510,1022,1285


### Tarihsel SMOTE tablosu — önceki yöntem
Yeni kat-içi akışın sonucu değildir.
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Model / Dengeleme</th>
      <th>OOF_AP</th>
      <th>ROC_AUC</th>
      <th>F1</th>
      <th>Precision</th>
      <th>Recall</th>
      <th>FP</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Logistic Regression (Doğal)</td>
      <td>0.4125</td>
      <td>0.8218</td>
      <td>0.4303</td>
      <td>0.3874</td>
      <td>0.4839</td>
      <td>2190</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Logistic Regression + SMOTE</td>
      <td>0.3921</td>
      <td>0.8177</td>
      <td>0.4230</td>
      <td>0.4160</td>
      <td>0.4301</td>
      <td>1728</td>
    </tr>
    <tr>
      <th>2</th>
      <td>XGBoost (Doğal)</td>
      <td>0.6058</td>
      <td>0.8985</td>
      <td>0.5599</td>
      <td>0.5889</td>
      <td>0.5335</td>
      <td>1066</td>
    </tr>
    <tr>
      <th>3</th>
      <td>XGBoost + SMOTE</td>
      <td>0.5754</td>
      <td>0.8893</td>
      <td>0.5346</td>
      <td>0.5137</td>
      <td>0.5573</td>
      <td>1510</td>
    </tr>
    <tr>
      <th>4</th>
      <td>LightGBM (Doğal)</td>
      <td>0.6142</td>
      <td>0.9010</td>
      <td>0.5680</td>
      <td>0.5779</td>
      <td>0.5584</td>
      <td>1167</td>
    </tr>
    <tr>
      <th>5</th>
      <td>LightGBM + SMOTE</td>
      <td>0.6048</td>
      <td>0.9006</td>
      <td>0.5548</td>
      <td>0.5853</td>
      <td>0.5273</td>
      <td>1069</td>
    </tr>
  </tbody>
</table>
</div>